<a href="https://colab.research.google.com/github/JUNNAY1188/Chronic-Kidney-Disease/blob/main/Chronic_Kidney_Disease.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# === 1) Import Libraries ===
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# === 2) Load Dataset ===
df = pd.read_csv("/content/kidney_disease.csv")

# Drop id column if present
if "id" in df.columns:
    df = df.drop(columns=["id"])

# === 3) Detect target column automatically ===
possible_targets = ["class", "classification"]
target_col = None
for col in possible_targets:
    if col in df.columns:
        target_col = col
        break

if target_col is None:
    raise ValueError("Target column not found! Please check dataset columns.")

print(f"Target column detected: {target_col}")

# === 4) Handle Missing Values ===
categorical_cols = df.select_dtypes(include=["object"]).columns.drop(target_col)
numeric_cols = df.select_dtypes(exclude=["object"]).columns

# Fill categorical with mode
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# Fill numeric with median
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# === 5) Encode categorical features ===
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# Encode target separately
target_encoder = LabelEncoder()
df[target_col] = target_encoder.fit_transform(df[target_col])

# === 6) Split features/target ===
X = df.drop(target_col, axis=1)
y = df[target_col]

# === 7) Train-test split ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# === 8) Scale numeric features ===
scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

# === 9) Train Model ===
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, y_train)

# === 10) Evaluate ===
y_pred = clf.predict(X_test)

print("\n=== Model Evaluation ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# === 11) Top 10 Feature Importances ===
importances = clf.feature_importances_
indices = np.argsort(importances)[-10:][::-1]

print("\n=== Top 10 Important Features ===")
for i, idx in enumerate(indices, 1):
    print(f"{i}. {X.columns[idx]}: {importances[idx]:.4f}")

# === 12) Predictions on test data ===
print("\n=== Sample Predictions (first 20 test cases) ===")
for true_val, pred in zip(y_test[:20], y_pred[:20]):
    status = "CKD (Disease)" if pred == 1 else "Not CKD"
    print(f"True: {true_val} → Predicted: {status}")

# === 13) Function for manual prediction ===
def predict_new(patient_data: dict):
    df_input = pd.DataFrame([patient_data])

    # Ensure column order matches training
    df_input = df_input[X.columns]

    # Scale numeric features
    df_input[numeric_cols] = scaler.transform(df_input[numeric_cols])

    prediction = clf.predict(df_input)[0]
    return "CKD (Disease)" if prediction == 1 else "Not CKD"

# === 14) Example Manual Prediction ===
example_patient = {
    "age": 40, "bp": 80, "sg": 1.02, "al": 1, "su": 0, "rbc": 1, "pc": 1,
    "pcc": 0, "ba": 0, "bgr": 120, "bu": 36, "sc": 1.2, "sod": 138, "pot": 4.5,
    "hemo": 15, "pcv": 44, "wc": 6700, "rc": 5.2, "htn": 1, "dm": 0, "cad": 0,
    "appet": 1, "pe": 0, "ane": 0
}
print("\nManual Prediction Example:", predict_new(example_patient))


Target column detected: classification

=== Model Evaluation ===
Accuracy: 0.9875
Confusion Matrix:
 [[50  0]
 [ 1 29]]

Classification Report:
               precision    recall  f1-score   support

           0       0.98      1.00      0.99        50
           2       1.00      0.97      0.98        30

    accuracy                           0.99        80
   macro avg       0.99      0.98      0.99        80
weighted avg       0.99      0.99      0.99        80


=== Top 10 Important Features ===
1. sc: 0.1800
2. hemo: 0.1773
3. pcv: 0.1277
4. sg: 0.1093
5. al: 0.0738
6. rc: 0.0621
7. dm: 0.0555
8. htn: 0.0393
9. bu: 0.0342
10. sod: 0.0275

=== Sample Predictions (first 20 test cases) ===
True: 0 → Predicted: Not CKD
True: 0 → Predicted: Not CKD
True: 2 → Predicted: Not CKD
True: 2 → Predicted: Not CKD
True: 2 → Predicted: Not CKD
True: 0 → Predicted: Not CKD
True: 0 → Predicted: Not CKD
True: 0 → Predicted: Not CKD
True: 2 → Predicted: Not CKD
True: 0 → Predicted: Not CKD
True: 0